In [1]:
EXPERIMENTAL_ID="product_search_recall_analyzer"
START_DATE="2025-07-09"

In [2]:
import requests
import pandas as pd

url = "http://dic.summerfarm.net/synonym.txt"
keywords_to_include=requests.get(url).text.split("\n")
keywords_to_include=[keyword.strip().split(",") for keyword in keywords_to_include]

indivitual_keywords_to_include=[]
for arr in keywords_to_include:
    for keyword in arr:
        indivitual_keywords_to_include.append(keyword)

print(indivitual_keywords_to_include[:10])

indivitual_keywords_to_include_df=pd.DataFrame(indivitual_keywords_to_include,columns=["keyword_synonym"])
indivitual_keywords_to_include_df.head(5)

['罐头', '荔枝罐头', '西柚粒罐头', '葡萄罐头', '丸子', '鸡肉丸', '牛肉丸', '黑巧', '黑巧克力', '白巧']


,keyword_synonym
0,罐头
1,荔枝罐头
2,西柚粒罐头
3,葡萄罐头
4,丸子


In [3]:
from odps_client import get_odps_sql_result_as_df
from datetime import datetime, timedelta

daily_high_search_volume_theshold = 470 / 14
daily_low_search_volume_theshold = 2

last_n_days = 60
ds_yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
last_n_days_ago = (datetime.now() - timedelta(days=last_n_days)).strftime("%Y%m%d")

        #         ,CASE
        #     WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > {last_n_days*daily_high_search_volume_theshold} THEN '高频搜索词'
        #     WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) <= {last_n_days*daily_low_search_volume_theshold} THEN '低频搜索词'
        #     ELSE '中频搜索词'
        #  END

top_query = f"""
SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS std_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.5) AS p50_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.75) AS p75_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.9) AS p90_click_index
        ,"不区分频次" AS 搜索频次标签
        ,COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) as ctr_uv
        ,COUNT(distinct ds) as 有搜索天数
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > 0.25 THEN '高点击率词'
            ELSE '低点击率词'
         END AS 点击率标签
FROM    summerfarm_tech.app_log_search_detail_di
WHERE   ds BETWEEN '{last_n_days_ago}' and '{ds_yesterday}'
GROUP BY query
ORDER BY searched_users DESC;
"""

top_query_df = get_odps_sql_result_as_df(sql=top_query)
top_query_df.head(20)

2025-07-28 10:17:38 - INFO - Thread count: 20
2025-07-28 10:18:12 - INFO - Tunnel session created: <InstanceDownloadSession id=202507281018119bde321a05ca081d project_name=summerfarm_ds instance_id=20250728021739246g03piyth1l1>
2025-07-28 10:18:13 - INFO - sql:

SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS s

,query,searched_users,search_cnt,click_cnt,avg_click_index,max_click_index,min_click_index,std_click_index,p50_click_index,p75_click_index,p90_click_index,搜索频次标签,ctr_uv,有搜索天数,点击率标签
0,芒果,12309,221008,74428,5.1,97.0,0.0,5.9,3.0,7.0,13.0,不区分频次,0.336766,60,高点击率词
1,牛奶,11792,110915,40132,8.8,290.0,0.0,12.4,5.0,12.0,24.0,不区分频次,0.361827,60,高点击率词
2,柠檬,10989,119834,38224,5.9,161.0,0.0,6.3,4.0,8.0,12.0,不区分频次,0.318975,60,高点击率词
3,草莓,9833,157962,52204,4.1,392.0,0.0,6.9,2.0,5.0,9.0,不区分频次,0.330485,60,高点击率词
4,安佳,8942,73086,23767,2.9,211.0,0.0,6.9,0.0,3.0,8.0,不区分频次,0.325192,60,高点击率词
5,奶油,7953,84719,19631,18.6,226.0,0.0,23.8,9.0,27.0,49.0,不区分频次,0.231719,60,低点击率词
6,蓝莓,7669,92508,30531,2.7,363.0,0.0,5.2,1.0,4.0,7.0,不区分频次,0.330036,60,高点击率词
7,西瓜,6415,107971,31633,4.7,239.0,0.0,6.8,3.0,6.0,12.0,不区分频次,0.292977,60,高点击率词
8,荔枝,6157,62107,18484,2.7,212.0,0.0,6.8,1.0,3.0,7.0,不区分频次,0.297615,60,高点击率词
9,黄油,5837,46638,9969,13.9,263.0,0.0,15.5,8.0,21.0,33.0,不区分频次,0.213753,60,低点击率词


In [4]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd

# 设置pandas显示选项以展示更多内容
pd.set_option("display.max_rows", 100)  # 显示最多100行
pd.set_option("display.max_columns", None)  # 显示所有列
pd.set_option("display.width", 1000)  # 设置显示宽度
pd.set_option("display.max_colwidth", 100)  # 设置列最大宽度

import sqlite3


def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    """
    从SLS(Simple Log Service)获取指定日期的用户变体数据。

    Args:
        day (datetime): 要获取数据的日期。
        check_if_local_exist (bool): 是否检查本地数据库中是否已存在数据，默认为True。

    Returns:
        pd.DataFrame: 包含用户变体数据的DataFrame。
    """
    # 构建数据库文件名和表名
    db_file_name = f"./data/search_ab_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    # 连接到SQLite数据库
    conn = sqlite3.connect(db_file_name)

    # 如果设置为检查本地数据
    if check_if_local_exist:
        try:
            # 尝试从数据库中读取数据
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            # 关闭数据库连接
            conn.close()
            # 返回读取的数据
            return df
        except pd.io.sql.DatabaseError:
            # 如果表不存在，则忽略错误
            pass

    # 构建SLS查询语句
    query = f"""
type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{{digit}}') as api,
    pageName as page_name,
    regexp_extract(experiment_item, '"experimentId":"([^"]+)"', 1) as experiment_id,
    type,
    uid,
    date_format(__time__, '%Y%m%d') as ds,
    count(1) as search_times,
    array_join(array_sort(array_agg(distinct regexp_extract(experiment_item, '"variantId":"([^"]+)"', 1))),',') as variant_list
FROM log, 
UNNEST(regexp_extract_all(json_extract_scalar(ai, '$.qh.xm-ab-exp'), '\{{[^}}]+\}}')) as t(experiment_item)
WHERE experiment_item LIKE '%"experimentId":"{EXPERIMENTAL_ID}"%'
GROUP BY 1,2,3,4,5,6
LIMIT 1000000
"""
    # 设置查询的起始时间和结束时间
    print(query)
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    # 从SLS获取数据
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",  # 指定SLS项目
        logstore="xm-mall",  # 指定SLS日志库
        from_time=from_time,  # 指定查询起始时间
        to_time=to_time,  # 指定查询结束时间
    )

    # 将search_times列中的缺失值填充为1，并转换为整数类型
    _df["search_times"] = _df["search_times"].fillna(1).astype(int)
    # 将variant_list列中的缺失值填充为"none"
    _df["variant_list"] = _df["variant_list"].fillna("none")

    # 如果DataFrame不为空
    if not _df.empty:
        # 删除不需要的列
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        # 将数据写入SQLite数据库，如果表已存在则替换
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    # 关闭数据库连接
    conn.close()
    # 返回数据
    return _df


# 创建一个空的DataFrame来存储所有日期的用户变体数据
all_user_variant_df = pd.DataFrame()
# 设置起始日期和结束日期
start_date = datetime.strptime(START_DATE, "%Y-%m-%d")
end_date = datetime.now()
# 从起始日期开始循环，直到结束日期
current_date = start_date
while current_date <= end_date:
    # 检查是否是今天
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    # 如果是今天,则跳过，因为今天的数据可能不完整
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    # 获取当前日期的用户变体数据
    df = get_user_variant_of_date_from_sls(current_date, check_if_local_exist=True)
    # 将当前日期的数据添加到总的DataFrame中
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    # 日期增加一天
    current_date += timedelta(days=1)

# 显示前10行数据
all_user_variant_df.head(10)


type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{digit}') as api,
    pageName as page_name,
    regexp_extract(experiment_item, '"experimentId":"([^"]+)"', 1) as experiment_id,
    type,
    uid,
    date_format(__time__, '%Y%m%d') as ds,
    count(1) as search_times,
    array_join(array_sort(array_agg(distinct regexp_extract(experiment_item, '"variantId":"([^"]+)"', 1))),',') as variant_list
FROM log, 
UNNEST(regexp_extract_all(json_extract_scalar(ai, '$.qh.xm-ab-exp'), '\{[^}]+\}')) as t(experiment_item)
WHERE experiment_item LIKE '%"experimentId":"product_search_recall_analyzer"%'
GROUP BY 1,2,3,4,5,6
LIMIT 1000000

即将获取数据: =====> 2025-07-21 00:00:00 2025-07-21 23:59:59.999999 xm-mall: 
type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+',
>=====数条数:7022

type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{digit}') as api,
    pageName

,api,page_name,experiment_id,type,uid,ds,search_times,variant_list
0,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,287515,20250709,1,V1
1,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,384443,20250709,1,V4
2,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,340322,20250709,1,V2
3,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,529830,20250709,5,V2
4,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,102271,20250709,2,V4
5,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,527851,20250709,1,V2
6,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,351763,20250709,7,V3
7,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,528342,20250709,2,V3
8,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,346343,20250709,1,V4
9,/mall/sku/page,/search/goods-new,product_search_recall_analyzer,a,207309,20250709,2,V3


In [5]:
import pandasql
stats=pandasql.sqldf("""select ds,variant_list,count(distinct uid) unique_user 
                     from all_user_variant_df group by ds,variant_list order by ds desc,variant_list""")

display(stats)

,ds,variant_list,unique_user
0,20250727,V1,1638
1,20250727,V2,1735
2,20250727,V3,1643
3,20250727,V4,1667
4,20250726,V1,1720
5,20250726,V2,1728
6,20250726,V3,1657
7,20250726,V4,1726
8,20250725,V1,1977
9,20250725,V2,1865


In [6]:
# idx:4,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:69.5,pdid:702,stock:10000,ext:cross;idx:5,name:徐州奶油草莓 280G*1盒/一级/4*6/ ,pid:goods,sku:5442468073,salePrice:16.5,pdid:702,stock:10000,ext:cross;idx:6,name:徐州奶油草莓 净重2.8-3斤/一级/单果10g+/ 10盒,pid:goods,sku:5442468518,salePrice:74.5,pdid:702,stock:10000,ext:cross

view_query = """
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,search_query,type,uid,ds
from(
select uid,date_format(__time__, '%Y%m%d') ds,bid_list.sku_item,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)
having pid = 'goods'
"""


def get_user_sku_view_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_view.db"
    table_name = f"user_sku_view_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=view_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_view_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_view_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_view_df = pd.concat([all_user_sku_view_df, df], ignore_index=True)
    current_date += timedelta(days=1)

即将获取数据: =====> 2025-07-21 00:00:00 2025-07-21 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:164781
即将获取数据: =====> 2025-07-22 00:00:00 2025-07-22 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:174879
即将获取数据: =====> 2025-07-23 00:00:00 2025-07-23 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:174578
即将获取数据: =====> 2025-07-24 00:00:00 2025-07-24 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:185643
即将获取数据: =====> 2025-07-25 00:00:00 2025-07-25 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:184249
即将获取数据: =====> 2025-07-26 00:00:00 2025-07-26 23:59:59.

In [7]:
from sls_client import get_sls_raw_data_by_query

click_query = """
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_item, 'name:([^,]+)', 1)) AS name,
  coalesce(sku,regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1)) AS sku,
  coalesce(pid,regexp_extract(sku_item, 'pid:([^,]+)', 1)) AS pid,
  coalesce(pdid,regexp_extract(sku_item, 'pdid:(\d+)', 1)) AS pdid,bid,
ds,search_query,type,uid,page_name,sku_item,coalesce(linkInfo,url)linkInfo from(
select uid,date_format(__time__, '%Y%m%d') ds,replace(replace(split_part(url_decode(split_part(url,'#/',2)),'?',2),'=',':'),'&',',') url,
bid_list.sku_item,pageName as page_name,bid,idx,name,sku,pid,pdid,linkInfo,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 1000000)"""

# click_query = "type:cl and pageName:/search/goods"


def get_user_sku_click_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_click.db"
    table_name = f"user_sku_click_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_raw_data_by_query(
        query=click_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(
            columns=[
                "__source__",
                "__time__",
                "userAgent",
                "url",
                "__topic__",
                "__tag__:__client_ip__",
                "__tag__:__receive_time__",
                "__time_ns_part__",
            ],
            inplace=True,
            errors="ignore",
        )
        _df["ds"] = day.strftime("%Y%m%d")
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_click_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_click_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_click_df = pd.concat([all_user_sku_click_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_click_df.head(10)

即将获取数据: =====>from_time:2025-07-21 00:00:00, to_time:2025-07-21 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_i
>=====数据条数:29541
即将获取数据: =====>from_time:2025-07-22 00:00:00, to_time:2025-07-22 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_i
>=====数据条数:32643
即将获取数据: =====>from_time:2025-07-23 00:00:00, to_time:2025-07-23 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_i
>=====数据条数:32531
即将获取数据: =====>from_time:2025-07-24 00:00:00, to_time:2025-07-24 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extra

,idx,name,sku,pid,pdid,bid,ds,search_query,type,uid,page_name,sku_item,linkInfo
0,13,雀巢烘烤淡奶油 1L*1盒,15107342408,唤起购买,1550,undefined,20250709,淡奶油,cl,531672,/search/goods-new,undefined,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
1,null,加入购物车,15107342408,加购弹窗,1550,"name:加入购物车,pid:加购弹窗,sku:15107342408,pdid:1550,stock:16",20250709,淡奶油,cl,531672,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:15107342408,pdid:1550,stock:16","name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
2,0,蓝风车蓝米吉稀奶油 1L*12盒,L001S01R001,唤起购买,71,undefined,20250709,蓝风车,cl,238551,/search/goods-new,undefined,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
3,16,新鲜红树莓 125G*12盒/一级/标准规格,663785768,goods,562,"idx:16,name:新鲜红树莓 125G*12盒/一级/标准规格,pid:goods,sku:663785768,salePrice:302,pdid:562,stock:0,ext:cross",20250709,树莓,cl,145686,/search/goods-new,"idx:16,name:新鲜红树莓 125G*12盒/一级/标准规格,pid:goods,sku:663785768,salePrice:302,pdid:562,stock:0,ext:cross","name:Search,word:限定用户特价！,linkShadingWord:[object Object],isTiming:no"
4,0,安佳淡奶油 1L*12盒,N001S01R005,唤起购买,56,undefined,20250709,奶油,cl,77732,/search/goods-new,undefined,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
5,null,步进器,1043683426715,加购弹窗,null,undefined,20250709,原味松松,cl,68565,/search/goods-new,undefined,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
6,null,加入购物车,1043683426715,加购弹窗,6634,"name:加入购物车,pid:加购弹窗,sku:1043683426715,pdid:6634,stock:8",20250709,原味松松,cl,68565,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:1043683426715,pdid:6634,stock:8","name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
7,null,加入购物车,N001S01R005,加购弹窗,56,"name:加入购物车,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:1590",20250709,奶油,cl,77732,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:1590","name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
8,0,酷盖纯牛奶 1L*12盒,607330063305,唤起购买,9260,undefined,20250709,牛奶,cl,377222,/search/goods-new,undefined,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
9,0,酷盖纯牛奶 1L*12盒,607330063305,唤起购买,9260,undefined,20250709,牛奶,cl,430200,/search/goods-new,undefined,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"


In [8]:
import re

all_user_sku_click_explored = []
pattern = re.compile(r'idx:(?P<idx>\d+).*?name:(?P<name>[^,]+).*?pid:(?P<pid>[^,]+).*?sku:(?P<sku>[^,]+).*?pdid:(?P<pdid>[^,]+)')

for index, row in all_user_sku_click_df.iterrows():
    _dict = row.to_dict()
    search_query=_dict["search_query"]
    if not search_query or f"{search_query}" == "":
        search_query = _dict["linkInfo"]
        # 搜索pdName，如果没找到，则search_query为空字符串
        match = re.search(r'pdName:([^,]+)', search_query)
        search_query = match.group(1) if match else ""
    bid = _dict["bid"]
    for bid_item in bid.split(";"):
        sku_info = {}
        sku_info.update(_dict)
        sku_info["search_query"] = search_query
        sku_info["bid"] = bid_item
        if 'undefined' in bid_item:
            all_user_sku_click_explored.append(sku_info)
        else:
            try:
                idx = pdid = sku = pid = name = None
                
                idx_match = re.search(r'idx:(\d+)', bid_item)
                if idx_match:
                    idx = idx_match.group(1)
                    
                pdid_match = re.search(r'pdid:(\d+)', bid_item)
                if pdid_match:
                    pdid = pdid_match.group(1)
                    
                sku_match = re.search(r'sku:([\dA-Za-z]+)', bid_item)
                if sku_match:
                    sku = sku_match.group(1)
                    
                pid_match = re.search(r'pid:([^,]+)', bid_item)
                if pid_match:
                    pid = pid_match.group(1)
                    
                name_match = re.search(r'name:([^,]+)', bid_item)
                if name_match:
                    name = name_match.group(1)
                    
                sku_info.update({
                    "idx": idx,
                    "pdid": pdid, 
                    "sku": sku,
                    "pid": pid,
                    "name": name
                })
                all_user_sku_click_explored.append(sku_info)
            except Exception as e:
                print(e, bid_item)
                raise e

all_user_sku_click_explored_df = pd.DataFrame(all_user_sku_click_explored)
all_user_sku_click_explored_df[['bid','sku','name','idx','pid','pdid','linkInfo','search_query']].head(5)

,bid,sku,name,idx,pid,pdid,linkInfo,search_query
0,undefined,15107342408,雀巢烘烤淡奶油 1L*1盒,13,唤起购买,1550,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no",淡奶油
1,"name:加入购物车,pid:加购弹窗,sku:15107342408,pdid:1550,stock:16",15107342408,加入购物车,None,加购弹窗,1550,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no",淡奶油
2,undefined,L001S01R001,蓝风车蓝米吉稀奶油 1L*12盒,0,唤起购买,71,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no",蓝风车
3,"idx:16,name:新鲜红树莓 125G*12盒/一级/标准规格,pid:goods,sku:663785768,salePrice:302,pdid:562,stock:0,ext:cross",663785768,新鲜红树莓 125G*12盒/一级/标准规格,16,goods,562,"name:Search,word:限定用户特价！,linkShadingWord:[object Object],isTiming:no",树莓
4,undefined,N001S01R005,安佳淡奶油 1L*12盒,0,唤起购买,56,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no",奶油


In [9]:
print(all_user_variant_df.columns)
print(all_user_sku_view_df.columns)
print(all_user_sku_click_df.columns)

all_sku_view_data_df = all_user_sku_view_df[
    [
        "idx",
        "name",
        "sku",
        "pid",
        "pdid",
        "uid",
        "ds",
        "search_query",
        "type",
    ]
].merge(
    all_user_variant_df[["uid", "ds", "variant_list", "search_times"]],
    on=["uid", "ds"],
    how="left",
)


Index(['api', 'page_name', 'experiment_id', 'type', 'uid', 'ds', 'search_times', 'variant_list'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'search_query', 'type', 'uid', 'ds'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'bid', 'ds', 'search_query', 'type', 'uid', 'page_name', 'sku_item', 'linkInfo'], dtype='object')


In [10]:
all_user_sku_click_explored_df.groupby("pid").size().reset_index(name="count").sort_values(
    "count", ascending=False
).head(10)

,pid,count
3,加购弹窗,267895
4,唤起购买,228739
0,goods,140625
9,横版筛选栏,7957
1,mini榜单,4427
5,商品列表,3866
10,竖版筛选栏,474
2,null,23
6,提交订单,1
7,搜索图片资源位,1


In [11]:
user_click_with_variant_df = all_user_sku_click_explored_df.merge(
    all_user_variant_df[["uid", "ds", "variant_list"]],
    on=["uid", "ds"],
    how="left",
)

user_click_with_variant_df["action_type"] = user_click_with_variant_df.apply(
    lambda row: (
        "加入购物车"
        if row["pid"] == "加购弹窗" and row["name"] == "加入购物车"
        else "商品详情" if row["pid"] == "goods" else row["pid"]
    ),
    axis=1,
)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].fillna(-1)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].replace('null', -1).astype(int)

In [12]:
import pandasql

user_click_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,variant_list,ds,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,
count(case when action_type='商品详情' then 1 end) as 商品详情cnt,
count(case when action_type='加入购物车' then 1 end) as 加入购物车cnt,
count(case when action_type='唤起购买' and variant_list is not null then 1 end) as 唤起购买cnt,
count(case when (action_type='唤起购买' and variant_list is not null) or action_type='商品详情' then 1 end) as 总点击cnt,
count(case when (action_type='唤起购买' and variant_list is not null or action_type='商品详情') and idx>=0 and idx<=5 then 1 end) as 首屏总点击cnt,
round(avg(case when action_type='商品详情' or action_type='唤起购买' then idx end),1) as avg点击位置,
coalesce(max(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as max点击位置,
coalesce(min(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as min点击位置,
count(distinct sku) 点击SKU_cnt,
count(distinct search_query) 搜索词cnt                        
from user_click_with_variant_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
""")

user_click_with_variant_statistics_df.head(5)

,uid,variant_list,ds,搜索频次标签,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt
0,100027,V2,20250709,不区分频次,0,1,1,1,1,4.0,4,4,1,1
1,100027,V2,20250714,不区分频次,0,1,1,1,1,0.0,0,0,1,1
2,100027,V2,20250716,不区分频次,0,1,1,1,1,0.0,0,0,1,1
3,100041,V2,20250710,不区分频次,3,0,0,3,3,0.0,0,0,1,2
4,100041,V2,20250711,不区分频次,3,0,0,3,3,0.0,0,0,1,1


In [13]:
null_search_query_df = pandasql.sqldf(
    """select case when search_query is null or search_query = 'null' then 'null-search-query' else 'normal' end has_search_query,
                                    count(1) cnt from all_sku_view_data_df group by 1"""
)
null_search_query_df
# 约有2.3%的数据没有搜索词，这部分需要过滤掉
all_sku_view_data_df = all_sku_view_data_df[
    all_sku_view_data_df["search_query"] != "null"
]

In [14]:
all_sku_view_data_df["idx"] = all_sku_view_data_df["idx"].fillna(-1).astype(int)
user_view_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,ds,variant_list,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,min(a.search_query) sample_query,
count(1) as 商品查看cnt,
count(distinct sku) as 查看SKU_cnt,
count(distinct search_query) as 查看搜索词cnt,
max(idx) as max查看位置,
max(search_times) as 搜索翻页数cnt
from all_sku_view_data_df a
inner join indivitual_keywords_to_include_df c on a.search_query = c.keyword_synonym
left join top_query_df b on a.search_query = b.query
where c.keyword_synonym is not null
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
"""
)

print("unique sample_query:", user_view_with_variant_statistics_df['sample_query'].unique())

unique sample_query: ['草莓' '荔枝' '樱桃' '芒果' '西瓜' '凤梨' '葡萄' '梨' '橙子' '果酱' '牛奶' '芋圆' '椰浆' '青提'
 '牛油果' '葡萄汁' '猕猴桃' '马斯卡彭' '气泡水' '高筋粉' '果糖' '椰乳' '脆啵啵' '蓝莓' '艾恩摩尔' '冰淇淋'
 '淡奶油' '奶酪' '艾恩' '菠萝' '低筋粉' '大芋圆' '水蜜桃' '咖啡杯' '菠萝果酱' '巨峰' '台农' '鲜奶' '鲜牛奶'
 '巧克力' '西柚' '波波晶球' '杯子' '罐头' '晶球' '布丁' '芭乐' '杯盖' '低筋面粉' '圣女果' '脆波波' '提子'
 '桃子' '柚子' '苏打水' '夏黑' '水仙芒' '苹果' '红提' '糖浆' '杯' '夏橙' '奇异果' '燕麦奶' '葡萄罐头'
 '金煌芒' '番石榴' '埃及橙' '高筋面粉' '椰奶' '小芋圆' '冰激凌' '稀奶油' '澄善HPP速冻凤梨颗粒浆' '白巧克力'
 '阳光玫瑰' '砀山梨' '红柚' '水牛奶' '黑巧' '番茄' '雪梨' '原味' '薯条' '白巧' '水果' '车厘子' '土豆'
 '台农芒果' '澄善HPP速冻葡萄汁' '巨峰葡萄' '丸子' '黑巧克力' '澳洲脐橙' '草莓果酱' '红颜草莓' '果汁' '干酪'
 '海南水仙芒' '水牛乳' 'PET冷饮杯' '大青芒' '红西柚' '湖北夏橙' '贡梨' '脐橙' '蜜薯' '酒' '水牛' '香肠'
 '红葡萄' '伦晚橙' '果浆' '烤肠' '夏黑葡萄' '橘子' '植物奶' '盖子' '荔枝罐头' '塑料杯' '小台农' '桂花味'
 '红凯特芒' '南非橙' '雪克杯' '皇冠梨' '鲜牛乳' '金煌' '台农芒' '红富士' '薯角' '冷饮杯' '青凯特芒' '纽荷尔'
 '红薯' '金果' '西柚粒罐头' '鱼胶' '四川金煌芒' '牛肉丸' 'pet杯盖' '辣味' '葡萄酒' '晴王青提' '红心柚'
 '冷饮杯盖' '樱花味' '桂花风味']


In [15]:
all_data_df = user_view_with_variant_statistics_df.merge(
    user_click_with_variant_statistics_df, on=["uid", "ds", "variant_list","搜索频次标签"], how="left"
)
# 定义计数列名列表
count_columns = ["商品详情cnt", "加入购物车cnt", "唤起购买cnt", "首屏总点击cnt", "总点击cnt"]
# 遍历计数列
for col in count_columns:
    # 将空值填充为0并转换为整数类型
    all_data_df[col] = all_data_df[col].fillna(0).astype(int)

# 创建 "用户是否点击" 列
all_data_df["用户是否点击"] = (all_data_df["总点击cnt"] > 0).astype(int)

# 定义费率计算相关列名列表
rate_columns = [
    ("sku_click_rate", "商品详情cnt", "商品查看cnt"), # 商品详情点击率
    ("add_cart_rate", "加入购物车cnt", "商品查看cnt"), # 加入购物车率
    ("popup_click_rate", "唤起购买cnt", "商品查看cnt"), # 唤起购买率
]

# 遍历费率列
for rate_col, num_col, den_col in rate_columns:
    # 计算费率，空值填充0，保留5位小数，转换为浮点数
    all_data_df[rate_col] = (all_data_df[num_col] / all_data_df[den_col]).fillna(0).round(5).astype(float)

all_data_df.head(5)

,uid,ds,variant_list,搜索频次标签,sample_query,商品查看cnt,查看SKU_cnt,查看搜索词cnt,max查看位置,搜索翻页数cnt,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt,用户是否点击,sku_click_rate,add_cart_rate,popup_click_rate
0,,20250710,None,不区分频次,草莓,11,7,2,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
1,,20250715,None,不区分频次,荔枝,3,3,1,2,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
2,,20250717,None,不区分频次,樱桃,4,4,1,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
3,,20250718,None,不区分频次,芒果,24,4,1,2,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
4,,20250724,None,不区分频次,西瓜,3,3,1,2,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0


In [16]:
from IPython.core.display import HTML
import pandas as pd

css = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@4.0.0/dist/css/bootstrap.min.css" integrity="sha384-Gn5384xqQ1aoWXA+058RXPxPg6fy4IWvTNh0E263XmFcJlSAwiGgFAW/dAiS6JXm" crossorigin="anonymous">
<style type=\"text/css\">
#abTesting table,#abTesting .table {
    color: #333;
    font-family: unset;
    font-size: 12px;
    line-height: 1.5;
    width: 95vw;
    border-collapse:
    collapse; 
    border-spacing: 0;
    font-family: "SF Pro SC", "SF Pro Text", "SF Pro Icons", "PingFang SC", "Helvetica Neue", "Helvetica", "Arial", sans-serif;
}

body{
    padding-left: 1rem;
    padding-top: 1vh;
}

tr{
    border-bottom: 1px solid #C1C3D1;
}

tr:nth-child(even) {
    background-color: #F8F8F8;
}

#abTesting td, #abTesting th {
    /* border: 1px solid transparent; No more visible border */
    height: 30px;
    padding: 0.2rem;
}

#abTesting table tbody td,#abTesting .table tbody td{
    padding: 0.1rem .75rem;
    vertical-align: middle;
}

th {
    background-color: #DFDFDF; /* Darken header a bit */
    font-weight: bolder;
    font-size: larger;
    color: #000;
    text-align: center;
}
</style>
"""


def display_p_value_below_005(row: pd.Series, p_value_col_name: str = "p_value"):
    p_value = row[p_value_col_name]
    color = "black"
    if p_value is not None and p_value <= 0.05:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{p_value}</span>"""


def display_diff_to_v2(row: pd.Series, metric: str = "diff_to_v2%"):
    diff = row[metric]
    color = "green"
    if diff is not None and float(diff) > 0.0:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{diff:.4f} %</span>"""


def dataframe_to_html(df: pd.DataFrame, title: str):
    df_to_display = df.copy()

    df_to_display["p_value"] = df_to_display.apply(display_p_value_below_005, axis=1)
    df_to_display["diff_to_v2%"] = df_to_display.apply(display_diff_to_v2, axis=1)

    html_df = df_to_display.to_html(
        escape=False, index=False, classes="table dataframe"
    )
    html_content = f"""<html><head><meta charset="UTF-8">
    <meta name="title" content="{title}">
    {css}
    </head><body>
    <h2>{title}</h2>
    <h4>当P-value <= 0.05时表示实验结果统计学显著</h4>
    <span>统计学显著时，既可能表示该试验组是好于对照组，也可能是坏于对照组</span>
    <div id="abTesting">{html_df}</div></body></html>"""

    return html_content

In [17]:
import pandas as pd
from scipy.stats import ttest_ind


def calculate_p_values(
    df: pd.DataFrame,
    metric: str = "商品详情cnt",
    control_variant: str = "V2",
) -> pd.DataFrame:
    """
    Calculate p-values for each combination of category1 and page_name.
    Compares metric between control group (V1) and each of V2, V3, V4.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing A/B test data.
    - metric (str): The metric column to be analyzed (default is 'added_quantity').

    Returns:
    - pd.DataFrame: A DataFrame with category1, page_name, variant, p-value, and statistics columns.
    """
    p_values = []

    control = df[df["variant_list"] == control_variant][metric]
    control_avg = control.mean()

    for variant in ["V1", "V2", "V3", "V4"]:
        test_group = df[df["variant_list"] == variant]
        test = test_group[metric]

        # print(
        #     f'variant:{variant}, test_group ds length: {len(test_group["ds"].unique())}'
        # )
        if len(test_group["ds"].unique()) <= 0:
            continue

        # Calculate statistics
        stats = {
            "均值": round(test.mean(), 4),
            "std": round(test.std(), 4),
            f"diff_to_{control_variant}%".lower(): round(
                100.00 * (test.mean() - control_avg) / control_avg, 2
            ),
            "q50": test.quantile(0.5),
            "q75": test.quantile(0.75),
            "q90": test.quantile(0.9),
            "q95": test.quantile(0.95),
            "q97": test.quantile(0.97),
            "q99": test.quantile(0.99),
            "q995": test.quantile(0.995),
            "max": test.max(),
            "日均总数": round(test.sum() / len(test_group["ds"].unique())),
            "日均实验UV": round(
                len(test_group[["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日均转化UV": round(
                len(test_group[test_group[metric] > 0][["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日期范围": f"{test_group['ds'].min()}~{test_group['ds'].max()}".replace(
                "2025", ""
            ),
            "metric": metric,
        }

        # Ensure both groups have enough data for a valid t-test
        if len(control) > 1 and len(test) > 1:
            # Perform independent t-test
            stat, p_val = ttest_ind(control, test, equal_var=False, nan_policy="omit")
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": round(p_val, 4),
                    **stats,
                }
            )
            # print(f"stat:{stat}")
        else:
            # Not enough data for statistical testing
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": None,
                    **stats,
                }
            )

    return pd.DataFrame(p_values)

In [18]:
# Define the desired order for sorting
variant_order = ["V1", "V2", "V3", "V4"]


# Create a custom sort key function
def sort_key(variant):
    # Split variant by commas
    parts = variant.split(",")
    # Determine the order based on the first variant in the list
    if parts[0] in variant_order:
        return variant_order.index(parts[0])
    else:
        return len(variant_order)  # Place all other variants after V1, V2, V3, V4


metrics_list = [
    "sku_click_rate",
    "avg点击位置",
    "唤起购买cnt",
    "总点击cnt",
    "商品详情cnt",
    "加入购物车cnt",
    "商品查看cnt",
]
all_p_values_df = pd.DataFrame()
for metric in metrics_list:
    all_p_values_of_same_metric_df = pd.DataFrame()
    for label, group_df in all_data_df.groupby("搜索频次标签"):
        if label == "其他":
            # print("ignore 其他")
            continue
        p_values_df = calculate_p_values(
            group_df,
            metric=metric,
        )

        p_values_df["搜索频次"] = label

        p_values_df["variant_list"] = p_values_df["variant_list"].apply(
            lambda x: x if x in variant_order else "X_" + x
        )
        p_values_df = p_values_df.sort_values(
            by="variant_list", key=lambda x: x.map(sort_key)
        )
        p_values_df["variant_list"] = p_values_df["variant_list"].str.replace("X_", "")
        all_p_values_of_same_metric_df = pd.concat(
            [all_p_values_of_same_metric_df, p_values_df], ignore_index=True
        )
        all_p_values_df = pd.concat([all_p_values_df, p_values_df], ignore_index=True)

    title = f"搜索AB--{metric}_p-value分布-{p_values_df.iloc[0]['日期范围']}"

    html_content = dataframe_to_html(df=all_p_values_of_same_metric_df, title=title)
    file_path = f"./data/{title}.html"

    # 保存HTML到本地文件：
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"写入HTML成功！{file_path}")


title_all = f"搜索AB--指标全集_p-value分布-{all_p_values_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_p_values_df, title=title_all)
file_path = f"./data/{title_all}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
all_p_values_df

写入HTML成功！./data/搜索AB--sku_click_rate_p-value分布-0709~0727.html
写入HTML成功！./data/搜索AB--avg点击位置_p-value分布-0709~0727.html
写入HTML成功！./data/搜索AB--唤起购买cnt_p-value分布-0709~0727.html
写入HTML成功！./data/搜索AB--总点击cnt_p-value分布-0709~0727.html
写入HTML成功！./data/搜索AB--商品详情cnt_p-value分布-0709~0727.html
写入HTML成功！./data/搜索AB--加入购物车cnt_p-value分布-0709~0727.html
写入HTML成功！./data/搜索AB--商品查看cnt_p-value分布-0709~0727.html
写入HTML成功！./data/搜索AB--指标全集_p-value分布-0709~0727.html


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric,搜索频次
0,V1,0.0648,0.1002,0.3297,-6.41,0.0,0.10000,0.25000,0.400000,0.60000,1.125000,1.666670,15.0,87,869,393,0709~0727,sku_click_rate,不区分频次
1,V2,1.0000,0.1070,0.3505,0.00,0.0,0.10000,0.26923,0.428570,0.66667,1.279366,2.000000,12.0,96,894,415,0709~0727,sku_click_rate,不区分频次
2,V3,0.8677,0.1076,0.3320,0.58,0.0,0.10526,0.28571,0.447324,0.66667,1.250000,1.943812,13.0,93,861,400,0709~0727,sku_click_rate,不区分频次
3,V4,0.1229,0.1013,0.3268,-5.32,0.0,0.10000,0.25000,0.416670,0.62500,1.125000,1.666670,17.0,89,882,408,0709~0727,sku_click_rate,不区分频次
4,V1,0.2671,3.5013,5.8478,-2.19,2.0,4.00000,8.00000,12.140000,17.30000,27.300000,38.000000,153.6,2671,869,638,0709~0727,avg点击位置,不区分频次
5,V2,1.0000,3.5796,6.2721,0.00,2.0,4.00000,8.00000,12.500000,16.70000,26.490000,38.997500,191.0,2827,894,662,0709~0727,avg点击位置,不区分频次
6,V3,0.4373,3.5259,5.5828,-1.50,2.0,4.00000,8.30000,13.000000,17.55400,25.218000,34.300000,115.3,2669,861,628,0709~0727,avg点击位置,不区分频次
7,V4,0.5751,3.6201,6.1808,1.13,2.0,4.00000,8.20000,12.900000,18.00000,28.000000,37.877000,177.0,2805,882,648,0709~0727,avg点击位置,不区分频次
8,V1,0.0997,2.3584,2.6643,2.08,2.0,3.00000,5.00000,7.000000,8.00000,11.000000,14.000000,41.0,2050,869,648,0709~0727,唤起购买cnt,不区分频次
9,V2,1.0000,2.3105,2.6656,0.00,2.0,3.00000,5.00000,7.000000,8.00000,11.000000,13.000000,54.0,2065,894,663,0709~0727,唤起购买cnt,不区分频次


In [19]:
# 导入odps_client库中的两个函数：get_odps_sql_result_as_df 用于执行SQL查询并将结果作为DataFrame返回, write_pandas_df_into_odps 用于将pandas DataFrame写入ODPS表
from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps

# 定义分区规范字符串，使用当前日期（年-月-日格式）作为分区值
partition_spec = f"pt={datetime.now().strftime('%Y%m%d')}"

# 将DataFrame写入ODPS表
write_pandas_df_into_odps(
    df=all_user_variant_df,  # 要写入的DataFrame，这里是all_user_variant_df，包含了所有用户的变体信息
    table_name="summerfarm_ds.temp_search_ab_all_data_df",  # ODPS表名
    partition_spec=partition_spec,  # 分区规范
    overwrite=True,  # 如果表或分区已存在，是否覆盖
    lifecycle=30,  # 设置表的生命周期为30天
)

# 只分析哪些进入过搜索页面的用户的订单转化结果

# 将开始日期格式化为字符串（年-月-日）
start_date_str = start_date.strftime("%Y-%m-%d")

# 定义SQL查询字符串，用于获取用户订单数据.
# 这段SQL的目的是：从订单表和用户分流表中，根据用户ID和日期进行关联，
# 统计每个用户在不同实验变体下的订单总金额、订单数量和平均订单金额。
order_query = f"""
with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '{start_date_str} 00:00:00'
    AND     m_size = '单店'
),user_variants as (
    select ds as event_date,uid,variant_list
    from summerfarm_ds.temp_search_ab_all_data_df
    where pt=max_pt('summerfarm_ds.temp_search_ab_all_data_df')
)
select a.event_date,a.uid,a.variant_list,sum(b.total_price) as order_gmv,
    count(b.order_no) as order_cnt,round(sum(b.total_price)/count(b.order_no),2) as avg_order_gmv
from user_variants a
left join user_orders b on a.uid=b.m_id and a.event_date=b.order_date
group by a.event_date,a.uid,a.variant_list
"""

# 执行SQL查询并将结果作为DataFrame返回
user_orders_df = get_odps_sql_result_as_df(order_query)
# 显示DataFrame的前两行
user_orders_df.head(2)

2025-07-28 10:20:48 - INFO - DaraFrame字段合集:type,experiment_id,ds,pt,api_list,search_times,uid,variant_list,page_ame,create_time,page_name,api
2025-07-28 10:20:52 - INFO - Tunnel session created: <TableUploadSession id=20250728102052d3d9c20b06a05a38 project=summerfarm_ds table=temp_search_ab_all_data_df partition_spec=pt=20250728>
2025-07-28 10:21:03 - INFO - 成功写入odps:summerfarm_ds.temp_search_ab_all_data_df, partition_spec:pt=20250728, attemp:0
2025-07-28 10:21:21 - INFO - Tunnel session created: <InstanceDownloadSession id=20250728102120dd1b481a05cb7210 project_name=summerfarm_ds instance_id=20250728022103326gzyownkcp5>
2025-07-28 10:21:23 - INFO - sql:

with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '2025-07-09 00:00:00'
    AND     m_size = '单

,event_date,uid,variant_list,order_gmv,order_cnt,avg_order_gmv
0,20250709,100027,V2,349,1,349
1,20250709,10008,V2,None,0,None


In [20]:
user_orders_df["order_gmv"]=user_orders_df["order_gmv"].astype(float)
user_orders_df["avg_order_gmv"]=user_orders_df["avg_order_gmv"].astype(float)
user_orders_df["order_cnt"]=user_orders_df["order_cnt"].astype(int)
user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].describe()

count    81222.000000
mean       533.600926
std        935.407734
min          0.700000
25%        170.000000
50%        302.000000
75%        594.000000
max      57042.000000
Name: order_gmv, dtype: float64

In [21]:
print(
    f"所有订单的分布:\n",
    user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].quantile(
        [0.5, 0.75, 0.95, 0.99, 0.995, 0.996, 0.997, 0.999, 1]
    ),
)


# 这里排除哪些高单价的订单，否则对于数据分析来说不好处理。
user_orders_below_6k_df = user_orders_df[user_orders_df["order_gmv"] <= 6000]
print(
    "排除高单价的订单后的分布:\n",
    user_orders_below_6k_df["order_gmv"].quantile(
        [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ),
)

所有订单的分布:
 0.500      302.00000
0.750      594.00000
0.950     1610.00000
0.990     3540.00000
0.995     4998.21500
0.996     5558.00000
0.997     6143.92176
0.999    10724.66250
1.000    57042.00000
Name: order_gmv, dtype: float64
排除高单价的订单后的分布:
 0.01      44.900
0.05      78.000
0.25     169.430
0.50     301.000
0.75     590.000
0.95    1553.000
0.99    3109.328
Name: order_gmv, dtype: float64


In [22]:
user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].astype(
    float
)

user_orders_below_6k_df["avg_order_gmv"].fillna(0.0, inplace=True)
user_orders_below_6k_df["avg_order_gmv"] = user_orders_below_6k_df[
    "avg_order_gmv"
].astype(float)

user_orders_below_6k_df["category1"] = "ignore"
user_orders_below_6k_df["page_name"] = "ignore"

user_orders_during_ab_df = user_orders_below_6k_df[
    user_orders_below_6k_df["variant_list"].isin(["V1", "V2", "V3", "V4"])
]
user_orders_during_ab_df.rename(columns={"event_date": "ds"}, inplace=True)

all_order_pvalue_df = pd.DataFrame()
for metric in ["order_gmv", "avg_order_gmv", "order_cnt"]:
    gmv_df = calculate_p_values(df=user_orders_during_ab_df, metric=metric)
    display(gmv_df)
    all_order_pvalue_df = pd.concat([all_order_pvalue_df, gmv_df], ignore_index=True)

title = f"搜索AB--订单转化p-value分布-{all_order_pvalue_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_order_pvalue_df, title=title)
file_path = f"./data/{title}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
display(all_order_pvalue_df)

/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_3670/858576852.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_3670/858576852.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_orders_below_6k_df["order_g

,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.1222,501.0815,589.6334,1.82,303.0,588.75,1065.00,1564.3,1927.9742,3116.0880,4053.960,6000.0,531014,1060,1060,0709~0727,order_gmv
1,V2,1.0000,492.1089,581.5167,0.00,297.0,582.45,1052.40,1545.0,1900.0000,2999.5610,4018.805,6000.0,532876,1083,1083,0709~0727,order_gmv
2,V3,0.0174,506.0983,600.6830,2.84,304.0,600.00,1079.85,1541.0,1968.1660,3125.0900,4193.925,5959.0,529592,1046,1046,0709~0727,order_gmv
3,V4,0.2684,498.5220,590.8327,1.30,301.0,588.00,1046.20,1569.3,1950.9600,3123.8848,4039.800,6000.0,534442,1072,1072,0709~0727,order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.5594,402.3360,442.6980,0.65,255.0,505.000,844.000,1150.0,1481.960,2257.220,2814.3200,6000.0,426370,1060,1060,0709~0727,avg_order_gmv
1,V2,1.0000,399.7370,455.5575,0.00,249.0,500.000,829.175,1160.0,1504.981,2271.985,2853.6450,6000.0,432852,1083,1083,0709~0727,avg_order_gmv
2,V3,0.5094,402.7276,456.0549,0.75,255.0,504.405,850.000,1150.0,1450.000,2330.000,2880.4875,5950.0,421423,1046,1046,0709~0727,avg_order_gmv
3,V4,0.7610,398.3920,439.0326,-0.34,252.3,503.000,838.550,1129.6,1450.000,2208.440,2767.2400,6000.0,427097,1072,1072,0709~0727,avg_order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.1617,1.2755,0.6930,0.73,1.0,1.0,2.0,2.0,3.0,4.0,5.0,15,1352,1060,1060,0709~0727,order_cnt
1,V2,1.0000,1.2663,0.6365,0.00,1.0,1.0,2.0,2.0,3.0,4.0,4.0,11,1371,1083,1083,0709~0727,order_cnt
2,V3,0.0001,1.2931,0.7354,2.11,1.0,1.0,2.0,3.0,3.0,4.0,5.0,17,1353,1046,1046,0709~0727,order_cnt
3,V4,0.4009,1.2715,0.6122,0.41,1.0,1.0,2.0,2.0,3.0,4.0,4.0,8,1363,1072,1072,0709~0727,order_cnt


写入HTML成功！./data/搜索AB--订单转化p-value分布-0709~0727.html


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.1222,501.0815,589.6334,1.82,303.0,588.750,1065.000,1564.3,1927.9742,3116.0880,4053.9600,6000.0,531014,1060,1060,0709~0727,order_gmv
1,V2,1.0000,492.1089,581.5167,0.00,297.0,582.450,1052.400,1545.0,1900.0000,2999.5610,4018.8050,6000.0,532876,1083,1083,0709~0727,order_gmv
2,V3,0.0174,506.0983,600.6830,2.84,304.0,600.000,1079.850,1541.0,1968.1660,3125.0900,4193.9250,5959.0,529592,1046,1046,0709~0727,order_gmv
3,V4,0.2684,498.5220,590.8327,1.30,301.0,588.000,1046.200,1569.3,1950.9600,3123.8848,4039.8000,6000.0,534442,1072,1072,0709~0727,order_gmv
4,V1,0.5594,402.3360,442.6980,0.65,255.0,505.000,844.000,1150.0,1481.9600,2257.2200,2814.3200,6000.0,426370,1060,1060,0709~0727,avg_order_gmv
5,V2,1.0000,399.7370,455.5575,0.00,249.0,500.000,829.175,1160.0,1504.9810,2271.9850,2853.6450,6000.0,432852,1083,1083,0709~0727,avg_order_gmv
6,V3,0.5094,402.7276,456.0549,0.75,255.0,504.405,850.000,1150.0,1450.0000,2330.0000,2880.4875,5950.0,421423,1046,1046,0709~0727,avg_order_gmv
7,V4,0.7610,398.3920,439.0326,-0.34,252.3,503.000,838.550,1129.6,1450.0000,2208.4400,2767.2400,6000.0,427097,1072,1072,0709~0727,avg_order_gmv
8,V1,0.1617,1.2755,0.6930,0.73,1.0,1.000,2.000,2.0,3.0000,4.0000,5.0000,15.0,1352,1060,1060,0709~0727,order_cnt
9,V2,1.0000,1.2663,0.6365,0.00,1.0,1.000,2.000,2.0,3.0000,4.0000,4.0000,11.0,1371,1083,1083,0709~0727,order_cnt


In [23]:
all_p_values_df.to_csv(
    f"./data/搜索AB--所有指标p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)
all_order_pvalue_df.to_csv(
    f"./data/搜索AB--订单转化p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)